# JetRacer FA2026 - Speed Track (Option B 3 kenh) + Tune + Lai tay

Quy trinh chay Speed Track sau khi `git pull` tren Jetson Nano:

1. **Buoc 0 (lan dau sau pull)**: chay cell `setup_jetson_models.py` de tu dong build cac TensorRT FP16 engine (`fast`, `resse`, `fastscnn`) tu file `.onnx`.
2. **Buoc 1**: chon `MODEL_MODE = 'resse'` (Model #4) de thu. Doi sang `'fast'` de so sanh voi Model #1, hoac `'fastscnn'` de thu Model #5.
3. **Buoc 2**: chay cell **MO GIAO DIEN SPEED TRACK** -> bam **1. MO CAMERA**.
4. Dat xe vao lan duong, doi banner `MAT DUONG OK - BAM CHAY` roi bam **2. CHAY - BAM LINE**. Panel Option B hien mat duong xanh va tam vang.
5. Keo **LAI CUA GAT** de chinh muc boi lai trong cua. Chinh **NHA LAI RA CUA (s)** de chon thoi gian nha lai dan khi ra cua gat.
6. **MO CUA RA NGOAI** thu mot nhip lai ra mep ngoai truoc cua khi con duong trong. Gia tri 0 tat che do nay.

**Truoc moi lan Run All**: `Kernel > Restart & Clear Output`. Hai notebook mo camera cung luc se bao `Failed to create CaptureSession`.

**AN TOAN**

- Mo len la trang thai DUNG. Khong lenh nao xuong phan cung cho den khi bam **2. CHAY - BAM LINE** hoac **LAI TAY**.
- Option B can vung mat duong xanh noi den xe qua 5 frame lien tiep de bam **CHAY**. Ga tang khi co huong lai.
- **DUNG KHAN CAP** cat ga ngay. Camera dung hoac loi -> tu dong cat ga.
- Lan dau test tren xe moi: **KE BANH KHOI MAT DAT**.

## 0. Thiet lap & Build TensorRT FP16 engine (chay 1 lan sau `git pull`)

In [ ]:
%cd /home/jetson/JetsonRacer
# Tu dong kiem tra moi truong + build 2 engine FP16 tren Jetson Nano.
# Neu engine da build san va moi hon file .onnx thi script tu dong bo qua rat nhanh.
!python3 tools/setup_jetson_models.py

## 1. Chon model & Kiem tra thu muc, engine

In [ ]:
%cd /home/jetson/JetsonRacer

import os
import socket

# ===================== CHON MODEL BAM VACH =====================
# 'fast'     -> Option B #1: LaneTiny-Fast  (NHE NHAT - 56K params, 0 SE, uu tien FPS >= 25)
# 'resse'    -> Option B #4: LaneTiny-ResSE (can bang - 157K params, Lookahead tot nhat)
# 'fastscnn' -> Option B #5: FastSCNN-Lane (A/B test, 331K params)
# 'cv'       -> CV co dien (khong dung model)
MODEL_MODE = 'resse'
# ===============================================================

MODE_MAP = {
    'fast':     ('configs/cnn_option_b_fast.yaml',     'models/lane_option_b_fast.engine'),
    'resse':    ('configs/cnn_option_b.yaml',          'models/lane_option_b_resse.engine'),
    'fastscnn': ('configs/cnn_option_b_fastscnn.yaml', 'models/lane_option_b_fastscnn.engine'),
    'cv':       (None, None),
}
if MODEL_MODE not in MODE_MAP:
    raise ValueError('MODEL_MODE phai la fast | resse | fastscnn | cv')

CFG_OVERRIDE, ENGINE_PATH = MODE_MAP[MODEL_MODE]
OVERRIDES = [CFG_OVERRIDE] if CFG_OVERRIDE else []
DUNG_CNN = (CFG_OVERRIDE is not None)

required = [
    'tools/tune_lane_jupyter.py',
    'src/jetracer_baseline/tuning_ui.py',
    'src/jetracer_baseline/perception/lane.py',
    'src/jetracer_baseline/perception/shading.py',
    'configs/default.yaml',
]
if DUNG_CNN:
    required += [
        'src/jetracer_baseline/perception/lane_cnn.py',
        ENGINE_PATH,
    ] + OVERRIDES

missing = [p for p in required if not os.path.exists(p)]
print('Hostname:', socket.gethostname())
print('Thu muc:', os.getcwd())
print('Che do:', MODEL_MODE, '->', OVERRIDES)
print('Files:', 'OK' if not missing else 'THIEU ' + ', '.join(missing))
if missing:
    if ENGINE_PATH and ENGINE_PATH in missing:
        print()
        print('Chua co file engine. Chay cell 0 o tren hoac go trong Terminal:')
        print('  python3 tools/setup_jetson_models.py')
    raise IOError('Thieu file, xem danh sach o tren')

print('Shading da hieu chuan:', os.path.exists('configs/shading.yaml'))

if DUNG_CNN:
    import sys
    if 'src' not in sys.path:
        sys.path.insert(0, 'src')
    from jetracer_baseline.config import load_config
    from jetracer_baseline.perception import build_lane_detector
    _det = build_lane_detector(load_config('configs/default.yaml', OVERRIDES))
    print('Engine nap OK:', type(_det).__name__, '|', ENGINE_PATH)
    if hasattr(_det, 'close'):
        _det.close()
    del _det

## 2. Nap ma nguon giao dien

In [ ]:
import importlib
import tools.tune_lane_jupyter as T
importlib.reload(T)
launch = T.launch
print('Da nap code. Keo xuong cell MO GIAO DIEN SPEED TRACK de chay.')

## 3. MO GIAO DIEN SPEED TRACK  <-- BAM O CELL NGAY DUOI

Cac nut chinh:

| Nut | Tac dung |
|---|---|
| **1. MO CAMERA** | bat camera, hien mat duong xanh va tam vang |
| **2. CHAY - BAM LINE** | bam khi hien `MAT DUONG OK - BAM CHAY` |
| **DUNG** | dung binh thuong & chot file log CSV |
| **DUNG KHAN CAP** | cat ga ngay lap tuc |
| Cac slider ben phai | TOC DO thang, TOC DO trong cua, CUA SOM, LAI CUA GAT, NHA LAI RA CUA, TRE VAO CUA, NHAY VUNG CUA, GIAM TOC TRUOC CUA, GOC LAI TOI DA |

Kiem tra bang XANH LA `MODEL CNN (TRT) - models/lane_option_b_...engine` truoc khi bam.

**`driver_kind='nvidia'` = XE CHAY THAT.** Lan dau: KE BANH KHOI MAT DAT.
Muon chi xem detect ma banh khong quay thi doi thanh `'dryrun'`.

In [ ]:
try:
    ui.close()
except NameError:
    pass

from tools.tune_lane_jupyter import launch

try:
    OVERRIDES
except NameError:
    OVERRIDES = ['configs/cnn_option_b.yaml']
    print('Chua chay cell 1 -> tu dat OVERRIDES =', OVERRIDES)

# simple=True      -> giao dien Speed Track voi camera, nut CHAY va cac thanh dieu chinh
# simple=False     -> giao dien day du (them tab Tay cam + Thu data + Luu config)
# driver_kind      -> 'nvidia' XE CHAY THAT | 'dryrun' banh khong quay
ui = launch(driver_kind='nvidia', overrides=OVERRIDES, simple=True)

## 4. (Tuy chon) Chay Speed Track bang CLI khong can UI

Mac dinh duoc comment de khi ban bam **Run All**, notebook KHONG tu chay xe ngay va KHONG xung dot camera voi giao dien o tren. Muon dung CLI thi bo dau `#` va dong UI truoc.

In [ ]:
# !python3 -m src.jetracer_baseline.cli run --task speed --driver nvidia --override configs/cnn_option_b.yaml --override configs/slow.yaml --max-seconds 60 --record

## 5. Dong giao dien khi chay xong (chay rieng cell nay, KHONG de chay trong Run All)

In [ ]:
# Bo dau # va chay rieng cell nay khi muon DONG camera + giao dien:
# ui.close()